In [ ]:
#xgboost notebook

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

In [4]:
# Define file paths
file_path_train = '/content/drive/My Drive/Colab Notebooks/Data/credit-risk/'

In [5]:
# Load the dataset
df = pd.read_excel(file_path_train + 'orbis_dataset.xlsx')

df.replace(['n.a.', 'n.s.'], np.nan, inplace=True)
df.drop('Status', axis=1)
print(df.shape)  # Should give (total_rows, total_columns)


(5506, 256)


<ipython-input-5-7b1f9d19a5db>:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['n.a.', 'n.s.'], np.nan, inplace=True)


In [10]:
# Define features (X) and target variable (y)
X = df.drop('Status Updated', axis=1)  # Features
y = df['Status Updated']
y = np.where(df['Status Updated'] == 'Active', 1, 0)

# One-hot encode categorical features before train_test_split
X = pd.get_dummies(X, drop_first=True)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42) # random_state for reproducibility

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(3854, 5879) (3854,)
(1652, 5879) (1652,)


In [7]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

# Define the XGBoost regressor with GPU support
xgb_regressor = xgb.XGBRegressor(
    tree_method='hist',
    device='cuda',
    random_state=42
)

# Hyperparameter grid for XGBoost
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],  # Optional: you can also tune this
    'colsample_bytree': [0.8, 1.0]  # Optional: you can also tune this
}

# GridSearchCV using the pre-defined xgb_regressor
grid_search = GridSearchCV(estimator=xgb_regressor,  # Use the model with GPU support
                           param_grid=param_grid,
                           scoring='neg_mean_squared_error',
                           cv=5)


# Fit grid search
grid_search.fit(X_train, y_train)

# Get the best parameters and the best model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Output the best parameters
print(f"Best Parameters from GridSearchCV: {best_params}")

ValueError: 
All the 540 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
108 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/xgboost/core.py", line 726, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py", line 1143, in fit
    train_dmatrix, evals = _wrap_evaluation_matrices(
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py", line 603, in _wrap_evaluation_matrices
    train_dmatrix = create_dmatrix(


In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
import numpy as np

# Train the model using the best parameters from GridSearchCV
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model using Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

# Print the results
print(f"Optimized XGBoost MSE: {mse}")
print(f"Optimized XGBoost RMSE: {rmse}")

# Cross-validation for model performance evaluation
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
mse_cv = -cv_scores.mean()  # Negate since sklearn returns negative MSE
rmse_cv = np.sqrt(mse_cv)
# Compute MAE
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Absolute Error: {mae}")
print(f"Cross-validated MSE: {mse_cv}")
print(f"Cross-validated RMSE: {rmse_cv}")
